This script quantifies the proportion of pre-existing built-up area within each new housing development area using the World Settlement Footprint (WSF) 2015 dataset. It performs a sensitivity analysis of different WSF coverage thresholds, generates summary statistics and visualizations, and exports filtered datasets for candidate threshold values.

In [ ]:
import subprocess
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.mask import mask as rio_mask
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# =============================================================================
# CONFIGURATION
# =============================================================================

NHDA_PATH  = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas\New_Housing_Development_Areas_residential.gpkg"
WSF_DIR    = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\WSF\2015"
OUTPUT_DIR = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas"

COL_OUT    = "ratio_wsf2015"
N_BINS     = 40
DPI        = 150
QUANTILES  = [0.05, 0.10, 0.25, 0.33, 0.50, 0.66, 0.75, 0.90, 0.95]

WSF_BUILT  = 255   # pixel value = built-up

# =============================================================================
# HELPERS
# =============================================================================

def build_vrt(wsf_dir: Path) -> Path:
    """
    Creates a GDAL VRT that mosaics all WSF TIFs virtually.
    No data is loaded into RAM – pixels are read on demand per polygon.
    Falls back to osgeo.gdal if gdalbuildvrt is not on PATH.
    """
    tifs = sorted(wsf_dir.glob("*.tif"))
    if not tifs:
        raise FileNotFoundError(f"No TIF files found in {wsf_dir}")
    print(f"   {len(tifs)} WSF TIF(s) found  →  building VRT")

    vrt_path = wsf_dir / "wsf2015_mosaic.vrt"

    # --- primary: gdalbuildvrt via subprocess ---
    try:
        cmd = ["gdalbuildvrt", str(vrt_path)] + [str(t) for t in tifs]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        print(f"   VRT created via gdalbuildvrt: {vrt_path.name}")
        return vrt_path

    except (FileNotFoundError, RuntimeError) as e:
        print(f"   gdalbuildvrt not available ({e}), trying osgeo.gdal fallback...")

    # --- fallback: osgeo.gdal (bundled with QGIS / OSGeo4W) ---
    try:
        from osgeo import gdal
        vrt = gdal.BuildVRT(str(vrt_path), [str(t) for t in tifs])
        vrt.FlushCache()
        vrt = None
        print(f"   VRT created via osgeo.gdal: {vrt_path.name}")
        return vrt_path

    except ImportError:
        raise RuntimeError(
            "Neither gdalbuildvrt nor osgeo.gdal is available.\n"
            "Please run this script from an OSGeo4W / QGIS Python environment,\n"
            "or add gdalbuildvrt to your PATH."
        )


def wsf_ratio_for_polygon(geom, raster_ds, nodata):
    """
    Computes ratio_wsf2015 for a single polygon geometry.
    Returns NaN if the polygon does not overlap the raster or is too small.
    """
    try:
        arr, _ = rio_mask(raster_ds, [geom], crop=True, all_touched=False)
        arr = arr[0]  # first (and only) band

        valid = arr[arr != nodata] if nodata is not None else arr.flatten()

        if len(valid) == 0:
            return np.nan

        built = int((valid == WSF_BUILT).sum())
        return round(built / len(valid), 6)

    except Exception:
        return np.nan


# =============================================================================
# MAIN
# =============================================================================

def main():
    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)
    (output_path / "Thresholds").mkdir(exist_ok=True)

    # =========================================================================
    # STEP 1: Load NHDA polygons
    # =========================================================================
    print("=" * 60)
    print("STEP 1: Load NHDA residential polygons")
    print("=" * 60)
    gdf = gpd.read_file(NHDA_PATH)
    print(f"✓ {len(gdf):,} polygons loaded  |  CRS: {gdf.crs}")

    # =========================================================================
    # STEP 2: Build VRT over WSF 2015 tiles  (no RAM merge!)
    # =========================================================================
    print(f"\n{'=' * 60}")
    print("STEP 2: Build WSF 2015 VRT")
    print("=" * 60)
    vrt_path = build_vrt(Path(WSF_DIR))

    with rasterio.open(vrt_path) as probe:
        wsf_crs    = probe.crs
        wsf_nodata = probe.nodata
    print(f"   WSF CRS:    {wsf_crs}")
    print(f"   WSF nodata: {wsf_nodata}")

    # Reproject NHDA polygons to WSF CRS for masking
    gdf_wsf = gdf.to_crs(wsf_crs)

    # =========================================================================
    # STEP 3: Compute ratio_wsf2015 per polygon
    # =========================================================================
    print(f"\n{'=' * 60}")
    print("STEP 3: Compute WSF 2015 coverage ratio per polygon")
    print("=" * 60)

    ratios = []
    # VRT stays open for the whole loop but only reads pixels per polygon window
    with rasterio.open(vrt_path) as wsf_ds:
        for i, geom in enumerate(gdf_wsf.geometry):
            r = wsf_ratio_for_polygon(geom, wsf_ds, wsf_nodata)
            ratios.append(r)
            if (i + 1) % 500 == 0 or (i + 1) == len(gdf_wsf):
                print(f"   {i + 1:,} / {len(gdf_wsf):,} processed...")

    gdf[COL_OUT] = ratios
    n_valid = sum(1 for r in ratios if not np.isnan(r))
    n_nan   = len(ratios) - n_valid
    print(f"\n✓ ratio_wsf2015 computed: {n_valid:,} valid  |  {n_nan:,} NaN")

    # =========================================================================
    # STEP 4: Export GeoPackage with ratio_wsf2015
    # =========================================================================
    print(f"\n{'=' * 60}")
    print("STEP 4: Export GeoPackage with ratio_wsf2015")
    print("=" * 60)
    out_gpkg = output_path / "New_Housing_Development_Areas_residential_wsf2015.gpkg"
    gdf.to_file(out_gpkg, driver="GPKG",
                layer="New_Housing_Development_Areas_residential_wsf2015")
    print(f"✓ Saved: {out_gpkg.name}")

    # =========================================================================
    # STEP 5: Descriptive statistics + quantile table
    # =========================================================================
    print(f"\n{'=' * 60}")
    print("STEP 5: Descriptive statistics")
    print("=" * 60)
    s = gdf[COL_OUT].dropna()

    print(f"\n  N (valid):  {len(s):,}")
    print(f"  Mean:       {s.mean():.4f}")
    print(f"  Median:     {s.median():.4f}")
    print(f"  Std:        {s.std():.4f}")
    print(f"  Min:        {s.min():.4f}")
    print(f"  Max:        {s.max():.4f}")

    print("\n  Quantiles:")
    for q in QUANTILES:
        val     = s.quantile(q)
        n_kept  = (s <= val).sum()
        pct     = n_kept / len(s) * 100
        print(f"    Q{q * 100:4.0f}%  →  threshold = {val:.4f}  "
              f"| keep {n_kept:,} ({pct:.1f}%) polygons with ratio ≤ threshold")

    # =========================================================================
    # STEP 6: Candidate threshold table
    # =========================================================================
    print(f"\n{'=' * 60}")
    print("STEP 6: Candidate threshold analysis")
    print("=" * 60)

    candidate_thresholds = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.33, 0.50]
    header = f"  {'Threshold':>10} | {'Kept (n)':>10} | {'Kept (%)':>9} | {'Removed (n)':>12} | {'Removed (%)':>11}"
    print(f"\n{header}")
    print("  " + "-" * 60)
    for t in candidate_thresholds:
        kept    = (s <= t).sum()
        removed = (s > t).sum()
        print(f"  {t:>10.2f} | {kept:>10,} | {kept / len(s) * 100:>8.1f}% | "
              f"{removed:>12,} | {removed / len(s) * 100:>10.1f}%")

    # =========================================================================
    # STEP 7: Visualization – Histogram + CDF
    # =========================================================================
    print(f"\n{'=' * 60}")
    print("STEP 7: Plots")
    print("=" * 60)

    shown_thresholds = [0.05, 0.10, 0.20, 0.33, 0.50]
    colors_thresh    = ["#d7191c", "#fdae61", "#1a9641", "#74c476", "#a6d96a"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("WSF 2015 – Built-up Coverage per NHDA Polygon",
                 fontsize=13, fontweight="bold")

    # --- (a) Histogram ---
    ax = axes[0]
    ax.hist(s, bins=N_BINS, color="#2c7bb6", edgecolor="white", linewidth=0.4)
    for color, t in zip(colors_thresh, shown_thresholds):
        ax.axvline(t, color=color, linewidth=1.4, linestyle="--",
                   label=f"t = {t:.2f}  (keep {(s <= t).sum() / len(s) * 100:.0f}%)")
    ax.set_xlabel("ratio_wsf2015  (0 = not built-up · 1 = fully built-up)")
    ax.set_ylabel("Number of polygons")
    ax.set_title("Distribution of WSF 2015 ratio")
    ax.legend(fontsize=8, title="Candidate thresholds\n(keep ratio ≤ t)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

    # --- (b) Cumulative distribution ---
    ax2 = axes[1]
    sorted_vals = np.sort(s.values)
    cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
    ax2.plot(sorted_vals, cdf * 100, color="#2c7bb6", linewidth=1.5)
    for color, t in zip(colors_thresh, shown_thresholds):
        pct = (s <= t).mean() * 100
        ax2.axvline(t, color=color, linewidth=1.2, linestyle="--")
        ax2.axhline(pct, color=color, linewidth=0.8, linestyle=":", alpha=0.6)
        ax2.annotate(f"{pct:.0f}%", xy=(t, pct), xytext=(t + 0.01, pct - 3),
                     fontsize=7.5, color=color)
    ax2.set_xlabel("ratio_wsf2015")
    ax2.set_ylabel("Cumulative share of polygons (%)")
    ax2.set_title("Cumulative distribution (CDF)")
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))

    plt.tight_layout()
    out_png = output_path / "sensitivity_ratio_wsf2015.png"
    fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"✓ Plot saved: {out_png.name}")

    # =========================================================================
    # STEP 8: Export filtered GeoPackages per threshold
    # =========================================================================
    print(f"\n{'=' * 60}")
    print("STEP 8: Export filtered GeoPackages per threshold")
    print("=" * 60)

    thresh_dir        = output_path / "Thresholds"
    export_thresholds = [0.05, 0.10, 0.20]

    for t in export_thresholds:
        # keep polygons with ratio ≤ t  OR  NaN (no WSF coverage → don't discard)
        mask_keep    = gdf[COL_OUT].isna() | (gdf[COL_OUT] <= t)
        gdf_filtered = gdf[mask_keep].copy()
        fname        = f"NHDA_residential_wsf2015_max{int(t * 100):02d}pct.gpkg"
        layer_name   = f"wsf2015_max{int(t * 100):02d}pct"
        gdf_filtered.to_file(thresh_dir / fname, driver="GPKG", layer=layer_name)
        print(f"  t ≤ {t:.2f}  →  {len(gdf_filtered):,} polygons kept  →  {fname}")

    print("\n✓ All done.")
# =========================================================================
    # STEP 8: Sensitivity plot – kept vs. removed per threshold
    # =========================================================================
    print(f"\n{'=' * 60}")
    print("STEP 8: Sensitivity plot")
    print("=" * 60)

    thresholds   = [0.01, 0.05, 0.10, 0.15, 0.20, 0.25, 0.33, 0.50]
    kept_n       = [(s <= t).sum() for t in thresholds]
    removed_n    = [(s >  t).sum() for t in thresholds]
    kept_pct     = [k / len(s) * 100 for k in kept_n]
    removed_pct  = [r / len(s) * 100 for r in removed_n]
    labels       = [f"{t:.2f}" for t in thresholds]
    x            = np.arange(len(thresholds))
    bar_w        = 0.55

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("WSF 2015 Sensitivity Analysis – NHDA Residential Polygons\n"
                 f"(N = {len(s):,}  |  Mean = {s.mean():.3f}  |  Median = {s.median():.3f})",
                 fontsize=13, fontweight="bold")

    # ── (A) Histogram + threshold lines ──────────────────────────────────────
    ax = axes[0, 0]
    ax.hist(s, bins=N_BINS, color="#2c7bb6", edgecolor="white", linewidth=0.4, zorder=2)
    colors5 = ["#d7191c", "#fdae61", "#1a9641", "#74c476", "#a6d96a"]
    for color, t in zip(colors5, [0.05, 0.10, 0.20, 0.33, 0.50]):
        ax.axvline(t, color=color, linewidth=1.5, linestyle="--", zorder=3,
                   label=f"t={t:.2f}  ({(s<=t).sum()/len(s)*100:.0f}% kept)")
    ax.set_xlabel("ratio_wsf2015")
    ax.set_ylabel("Number of polygons")
    ax.set_title("(A) Distribution + candidate thresholds")
    ax.legend(fontsize=8, title="threshold  (% kept)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

    # ── (B) CDF ───────────────────────────────────────────────────────────────
    ax2 = axes[0, 1]
    sv  = np.sort(s.values)
    cdf = np.arange(1, len(sv) + 1) / len(sv)
    ax2.plot(sv, cdf * 100, color="#2c7bb6", linewidth=2)
    for color, t in zip(colors5, [0.05, 0.10, 0.20, 0.33, 0.50]):
        pct = (s <= t).mean() * 100
        ax2.axvline(t, color=color, linewidth=1.2, linestyle="--")
        ax2.axhline(pct, color=color, linewidth=0.8, linestyle=":", alpha=0.7)
        ax2.annotate(f"{pct:.0f}%", xy=(t, pct), xytext=(t + 0.01, pct - 4),
                     fontsize=8, color=color, fontweight="bold")
    ax2.set_xlabel("ratio_wsf2015")
    ax2.set_ylabel("Cumulative share of polygons (%)")
    ax2.set_title("(B) Cumulative distribution (CDF)")
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))

    # ── (C) Stacked bar: kept vs removed (absolute) ───────────────────────────
    ax3 = axes[1, 0]
    ax3.bar(x, kept_n,    bar_w, label="Kept",    color="#2c7bb6", zorder=2)
    ax3.bar(x, removed_n, bar_w, bottom=kept_n,   label="Removed", color="#d7191c", zorder=2)
    for i, (k, r) in enumerate(zip(kept_n, removed_n)):
        ax3.text(i, k / 2,       f"{k:,}",  ha="center", va="center",
                 fontsize=8, color="white", fontweight="bold")
        ax3.text(i, k + r / 2,   f"{r:,}",  ha="center", va="center",
                 fontsize=8, color="white", fontweight="bold")
    ax3.set_xticks(x)
    ax3.set_xticklabels(labels)
    ax3.set_xlabel("WSF 2015 ratio threshold  (keep ≤ t)")
    ax3.set_ylabel("Number of polygons")
    ax3.set_title("(C) Kept vs. Removed (absolute)")
    ax3.legend()
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    ax3.set_ylim(0, len(s) * 1.08)

    # ── (D) Line: % kept per threshold ───────────────────────────────────────
    ax4 = axes[1, 1]
    ax4.plot(thresholds, kept_pct, color="#2c7bb6", linewidth=2,
             marker="o", markersize=7, zorder=3)
    for t, pct, n in zip(thresholds, kept_pct, kept_n):
        ax4.annotate(f"{pct:.1f}%\n(n={n:,})", xy=(t, pct),
                     xytext=(t, pct + 1.8), ha="center", fontsize=7.5,
                     color="#2c7bb6", fontweight="bold")
    ax4.axhline(100, color="grey", linewidth=0.8, linestyle=":")
    ax4.set_xlabel("WSF 2015 ratio threshold  (keep ≤ t)")
    ax4.set_ylabel("Polygons kept (%)")
    ax4.set_title("(D) Share of polygons kept per threshold")
    ax4.set_ylim(0, 115)
    ax4.set_xlim(-0.01, 0.53)
    ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))

    plt.tight_layout()
    out_png = output_path / "sensitivity_ratio_wsf2015.png"
    fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"✓ Plot saved: {out_png.name}")

    print("\n✓ All done.")

if __name__ == "__main__":
    main()